In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze quality checks
# MAGIC Bronze is a RAW layer: it keeps the data as the API returned it. Business rules belong to Silver.
# MAGIC - **CRITICAL** (fails the task): table missing, table empty, null primary keys.
# MAGIC - **WARN** (log only, the job continues): everything else (dates order, state rules, row-count drift, backfill depth ...).
# MAGIC
# MAGIC Run AFTER `extract_data()`, `build_bronze_table()` and `run_all_incremental()`.

# COMMAND ----------

from datetime import datetime
from pyspark.sql import functions as F

BRONZE = "workspace.bronze"
VERBOSE = True               # False = print only CRITICAL / WARN lines
MIN_RATIO = 0.95             # below this ratio of the API count -> WARN
HARD_FAIL_BELOW_RATIO = 0.0  # optional: set e.g. 0.5 to make a HUGE drop critical (0.0 = disabled)
FRESH_HOURS = 26             # raw_ tables should have been rebuilt within this many hours

REPOS = {
    "vscode": "microsoft/vscode",
    "TypeScript": "microsoft/TypeScript",
    "PowerToys": "microsoft/PowerToys",
    "terminal": "microsoft/terminal",
    "PowerShell": "PowerShell/PowerShell",
    "runtime": "dotnet/runtime",
    "aspnetcore": "dotnet/aspnetcore",
}

# Row counts measured from the GitHub API (since 2010)
EXPECTED = {
    ("vscode", "commits"): 166530,     ("vscode", "pull_requests"): 67361,
    ("TypeScript", "commits"): 39469,  ("TypeScript", "pull_requests"): 19662,
    ("PowerToys", "commits"): 9739,    ("PowerToys", "pull_requests"): 8732,
    ("terminal", "commits"): 5046,     ("terminal", "pull_requests"): 5362,
    ("PowerShell", "commits"): 11575,  ("PowerShell", "pull_requests"): 10946,
    ("runtime", "commits"): 129267,    ("runtime", "pull_requests"): 57045,
    ("aspnetcore", "commits"): 57540,  ("aspnetcore", "pull_requests"): 23985,
}

critical_failures, warnings = [], []

# COMMAND ----------

# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------
def critical(name, ok, detail=""):
    """Critical check: a failure makes the task fail at the end."""
    if VERBOSE or not ok:
        print(("PASS     " if ok else "CRITICAL "), name, detail)
    if not ok:
        critical_failures.append(name)


def warn(name, ok, detail=""):
    """Soft check: printed and collected, never stops the job."""
    if VERBOSE or not ok:
        print(("PASS     " if ok else "WARN     "), name, detail)
    if not ok:
        warnings.append(name)


def warn_if_nonzero(label, name, value):
    warn(f"{label}: {name}", n(value) == 0, f"count={n(value)}")


def cnt(cond):
    """Count rows where cond is true (null counts as false)."""
    return F.sum(F.when(cond, 1).otherwise(0))


def n(x):
    return x or 0


def ts(col_name):
    """String / timestamp column -> timestamp."""
    return F.to_timestamp(F.col(col_name))


# A little tolerance so clock differences do not cause false alarms
FUTURE_LIMIT = F.current_timestamp() + F.expr("INTERVAL 1 DAY")


def load(label, table, hard=True):
    """Read a table. Missing table = CRITICAL (or WARN when hard=False)."""
    if not spark.catalog.tableExists(table):
        (critical if hard else warn)(f"{label}: table exists", False, table)
        return None
    return spark.table(table)


def run_agg(df, critical_exprs, anomaly_exprs, label):
    """
    One scan for everything. If an anomaly column is missing (API schema drift),
    fall back to the critical expressions only, so a non-key column can never
    crash the check. Returns (row, anomalies_ran).
    """
    try:
        return df.agg(*critical_exprs, *anomaly_exprs).first(), True
    except Exception as e:
        warn(f"{label}: anomaly checks could not run", False, str(e)[:200])
    try:
        return df.agg(*critical_exprs).first(), False
    except Exception as e:
        critical(f"{label}: key columns are readable", False, str(e)[:200])
        return None, False


def safe(label, name, fn):
    """Run an optional check. If it errors, log a WARN instead of crashing."""
    try:
        fn()
    except Exception as e:
        warn(f"{label}: {name} could not run", False, str(e)[:200])


def not_empty(label, s, hard=True):
    """Table must contain rows. Returns True when it does."""
    ok = s["rows"] > 0
    (critical if hard else warn)(f"{label}: table is not empty", ok, f"rows={s['rows']}")
    return ok


def check_row_count(label, s, expected):
    """Small drift = WARN. A huge drop = CRITICAL only if HARD_FAIL_BELOW_RATIO is set."""
    warn(f"{label}: row count close to API count", s["rows"] >= expected * MIN_RATIO,
         f"rows={s['rows']} need>={int(expected * MIN_RATIO)}")
    if HARD_FAIL_BELOW_RATIO > 0:
        critical(f"{label}: row count is not far below API count",
                 s["rows"] >= expected * HARD_FAIL_BELOW_RATIO,
                 f"rows={s['rows']} need>={int(expected * HARD_FAIL_BELOW_RATIO)}")


def check_incremental_state(label, pipeline, raw_table):
    """Watermark moved, incremental is caught up with raw_, and raw_ is fresh (all WARN)."""
    wm = watermarks.get(pipeline)
    warn(f"{label}: watermark advanced past initial value",
         wm is not None and wm > datetime(2010, 1, 1), f"watermark={wm}")

    r = spark.table(raw_table).agg(
        F.max("ingested_at").alias("raw_max"),
        ((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.max("ingested_at"))) / 3600).alias("age_h"),
    ).first()

    warn(f"{label}: incremental is caught up with raw_",
         wm is not None and r["raw_max"] is not None and wm >= r["raw_max"],
         f"watermark={wm} raw_max={r['raw_max']}")

    age = r["age_h"]
    warn(f"{label}: raw_ was rebuilt in the last {FRESH_HOURS}h",
         age is not None and age <= FRESH_HOURS,
         f"age_hours={age:.1f}" if age is not None else "no data")

# COMMAND ----------

# ---------------------------------------------------------------
# 1) Control table
# ---------------------------------------------------------------
print("===== control_table =====")
watermarks = {}
ct = load("control_table", f"{BRONZE}.control_table")           # missing = CRITICAL
if ct is not None:
    watermarks = {r["pipeline_name"]: r["last_watermark"] for r in ct.collect()}
    needed = [f"{r.lower()}_{e}" for r in REPOS for e in ("commits", "pull_requests")]
    missing_pipes = [p for p in needed if p not in watermarks]
    warn("control_table: all 14 pipelines exist", not missing_pipes, f"missing={missing_pipes}")
    warn("control_table: no duplicate pipeline rows",
         ct.count() == ct.select("pipeline_name").distinct().count())
    warn("control_table: no null watermarks",
         all(watermarks[p] is not None for p in needed if p in watermarks))

# COMMAND ----------

# ---------------------------------------------------------------
# 2) Commits
#    CRITICAL: table exists, not empty, no null sha
#    WARN: everything else
# ---------------------------------------------------------------
def check_commits(repo, full):
    label = f"{repo}/commits"
    inc = f"{BRONZE}.inc_{repo.lower()}_commits"
    raw = f"{BRONZE}.raw_{repo}_commits"
    inc_df = load(label, inc)
    if inc_df is None:
        return

    committed = ts("commit.committer.date")
    authored = ts("commit.author.date")

    critical_exprs = [
        F.count("*").alias("rows"),
        cnt(F.col("sha").isNull()).alias("null_keys"),
    ]
    anomaly_exprs = [
        F.countDistinct("sha").alias("distinct_keys"),
        cnt(~F.col("sha").rlike("^[0-9a-f]{40}$")).alias("bad_sha"),
        cnt(committed.isNull()).alias("null_committer_date"),
        cnt(authored.isNull()).alias("null_author_date"),
        cnt(F.col("commit.message").isNull()).alias("null_message"),
        cnt(committed > FUTURE_LIMIT).alias("future_dates"),
        cnt(authored > committed).alias("author_after_committer"),
        cnt(~F.lower(F.col("html_url")).contains(f"/{full.lower()}/commit/")).alias("wrong_repo"),
        F.min(committed).alias("oldest"),
    ]
    s, anomalies_ok = run_agg(inc_df, critical_exprs, anomaly_exprs, label)
    if s is None:
        return

    # ---- CRITICAL ----
    if not not_empty(label, s):
        return
    critical(f"{label}: no null sha (primary key)", n(s["null_keys"]) == 0,
             f"count={n(s['null_keys'])}")

    # ---- WARN ----
    check_row_count(label, s, EXPECTED[(repo, "commits")])
    if not anomalies_ok:
        return
    warn(f"{label}: no duplicates (rows == distinct sha)", s["rows"] == s["distinct_keys"],
         f"rows={s['rows']} distinct={s['distinct_keys']}")
    warn_if_nonzero(label, "sha is a valid 40-char hash", s["bad_sha"])
    warn_if_nonzero(label, "no null committer date", s["null_committer_date"])
    warn_if_nonzero(label, "no null author date", s["null_author_date"])
    warn_if_nonzero(label, "no commit dated in the future", s["future_dates"])
    warn_if_nonzero(label, "all commits belong to this repo (html_url)", s["wrong_repo"])
    warn_if_nonzero(label, "message is never null", s["null_message"])
    # author date after committer date can happen legitimately (rebases, --date)
    warn(f"{label}: author date <= committer date (<1% exceptions)",
         n(s["author_after_committer"]) <= 0.01 * s["rows"],
         f"count={n(s['author_after_committer'])}")
    warn(f"{label}: history goes back before 2024 (backfill worked)",
         s["oldest"] is not None and s["oldest"] < datetime(2024, 1, 1), f"oldest={s['oldest']}")

    # raw_ vs inc_ consistency (all WARN)
    def cross_check():
        if load(label, raw, hard=False) is None:
            return
        missing = (spark.table(raw).select("sha").distinct()
                   .join(inc_df.select("sha"), "sha", "left_anti").count())
        warn_if_nonzero(label, "every raw_ commit exists in inc_", missing)
        check_incremental_state(label, f"{repo.lower()}_commits", raw)
    safe(label, "raw_ vs inc_ check", cross_check)

# COMMAND ----------

# ---------------------------------------------------------------
# 3) Pull requests
#    CRITICAL: table exists, not empty, no null id
#    WARN: dates order, state rules, update/merge consistency, ...
#    (Dirty rows such as closed_at < created_at are kept as-is: Silver handles them.)
# ---------------------------------------------------------------
def check_pull_requests(repo, full):
    label = f"{repo}/pull_requests"
    inc = f"{BRONZE}.inc_{repo.lower()}_pull_requests"
    raw = f"{BRONZE}.raw_{repo}_pull_requests"
    inc_df = load(label, inc)
    if inc_df is None:
        return

    created, updated = ts("created_at"), ts("updated_at")
    closed, merged = ts("closed_at"), ts("merged_at")
    state = F.col("state")

    critical_exprs = [
        F.count("*").alias("rows"),
        cnt(F.col("id").isNull()).alias("null_keys"),
    ]
    anomaly_exprs = [
        F.countDistinct("id").alias("distinct_keys"),
        cnt(F.col("number").isNull()).alias("null_number"),
        cnt(created.isNull()).alias("null_created"),
        cnt(updated.isNull()).alias("null_updated"),
        # date order
        cnt(created > updated).alias("created_after_updated"),
        cnt(closed < created).alias("closed_before_created"),
        cnt(merged < created).alias("merged_before_created"),
        cnt(merged > closed).alias("merged_after_closed"),
        cnt((created > FUTURE_LIMIT) | (updated > FUTURE_LIMIT)).alias("future_dates"),
        # state consistency
        cnt(state.isNull() | ~state.isin("open", "closed")).alias("bad_state"),
        cnt((state == "open") & closed.isNotNull()).alias("open_but_has_closed_at"),
        cnt((state == "closed") & closed.isNull()).alias("closed_without_closed_at"),
        cnt(merged.isNotNull() & closed.isNull()).alias("merged_without_closed_at"),
        cnt(merged.isNotNull() & (state != "closed")).alias("merged_but_not_closed"),
        # belongs to this repo
        cnt(F.lower(F.col("base.repo.full_name")) != full.lower()).alias("wrong_repo"),
        F.min(created).alias("oldest"),
    ]
    s, anomalies_ok = run_agg(inc_df, critical_exprs, anomaly_exprs, label)
    if s is None:
        return

    # ---- CRITICAL ----
    if not not_empty(label, s):
        return
    critical(f"{label}: no null id (primary key)", n(s["null_keys"]) == 0,
             f"count={n(s['null_keys'])}")

    # ---- WARN ----
    check_row_count(label, s, EXPECTED[(repo, "pull_requests")])
    if not anomalies_ok:
        return
    warn(f"{label}: no duplicates (rows == distinct id)", s["rows"] == s["distinct_keys"],
         f"rows={s['rows']} distinct={s['distinct_keys']}")
    warn_if_nonzero(label, "no null PR number", s["null_number"])
    warn_if_nonzero(label, "no null created_at", s["null_created"])
    warn_if_nonzero(label, "no null updated_at", s["null_updated"])

    # Dates: one must come before the other
    warn_if_nonzero(label, "created_at <= updated_at", s["created_after_updated"])
    warn_if_nonzero(label, "closed_at >= created_at", s["closed_before_created"])
    warn_if_nonzero(label, "merged_at >= created_at", s["merged_before_created"])
    warn_if_nonzero(label, "merged_at <= closed_at", s["merged_after_closed"])
    warn_if_nonzero(label, "no PR dated in the future", s["future_dates"])

    # State vs dates
    warn_if_nonzero(label, "state is 'open' or 'closed'", s["bad_state"])
    warn_if_nonzero(label, "open PR has no closed_at", s["open_but_has_closed_at"])
    warn_if_nonzero(label, "closed PR has closed_at", s["closed_without_closed_at"])
    warn_if_nonzero(label, "merged PR has closed_at", s["merged_without_closed_at"])
    warn_if_nonzero(label, "merged PR is in state 'closed'", s["merged_but_not_closed"])
    warn_if_nonzero(label, "all PRs belong to this repo (base.repo)", s["wrong_repo"])
    warn(f"{label}: history goes back before 2024 (backfill worked)",
         s["oldest"] is not None and s["oldest"] < datetime(2024, 1, 1), f"oldest={s['oldest']}")

    # Update / merge consistency: inc_ should hold the LATEST version of every PR in raw_
    def cross_check():
        if load(label, raw, hard=False) is None:
            return
        raw_latest = (spark.table(raw).groupBy("id")
                      .agg(F.max(ts("updated_at")).alias("raw_updated")))
        inc_ts = inc_df.select("id", ts("updated_at").alias("inc_updated"))

        missing = raw_latest.select("id").join(inc_ts.select("id"), "id", "left_anti").count()
        warn_if_nonzero(label, "every raw_ PR exists in inc_", missing)

        both = raw_latest.join(inc_ts, "id")
        stale = both.where(F.col("inc_updated") < F.col("raw_updated")).count()
        warn_if_nonzero(label, "inc_ has the latest version (not older than raw_)", stale)
        newer = both.where(F.col("inc_updated") > F.col("raw_updated")).count()
        warn_if_nonzero(label, "inc_ is not newer than raw_", newer)

        check_incremental_state(label, f"{repo.lower()}_pull_requests", raw)
    safe(label, "raw_ vs inc_ check", cross_check)

# COMMAND ----------

# ---------------------------------------------------------------
# 4) Small tables (full overwrite): repo_data, languages, contributors, releases
#    CRITICAL: table exists and is not empty (releases may legitimately be empty -> WARN)
#    WARN: everything else
# ---------------------------------------------------------------
def check_small_tables(repo, full):
    # ---- repo_data: one row describing the repo ----
    label = f"{repo}/repo_data"
    df = load(label, f"{BRONZE}.raw_{repo}_repo_data")
    if df is not None:
        s, ok = run_agg(df, [F.count("*").alias("rows")], [
            cnt(F.col("id").isNull()).alias("null_id"),
            cnt(ts("created_at").isNull()).alias("null_created"),
            cnt(F.col("stargazers_count") < 0).alias("negative_stars"),
            cnt(F.lower(F.col("full_name")) != full.lower()).alias("wrong_repo"),
        ], label)
        if s is not None and not_empty(label, s) and ok:
            warn(f"{label}: exactly one row", s["rows"] == 1, f"rows={s['rows']}")
            warn_if_nonzero(label, "id is not null", s["null_id"])
            warn_if_nonzero(label, "created_at is not null", s["null_created"])
            warn_if_nonzero(label, "stargazers_count >= 0", s["negative_stars"])
            warn_if_nonzero(label, "full_name matches this repo", s["wrong_repo"])

    # ---- languages: language -> bytes ----
    label = f"{repo}/languages"
    df = load(label, f"{BRONZE}.raw_{repo}_languages")
    if df is not None:
        s, ok = run_agg(df, [F.count("*").alias("rows")], [
            F.countDistinct("language").alias("distinct_languages"),
            cnt(F.col("language").isNull()).alias("null_language"),
            cnt(F.col("bytes") <= 0).alias("non_positive_bytes"),
        ], label)
        if s is not None and not_empty(label, s) and ok:
            warn(f"{label}: one row per language", s["rows"] == s["distinct_languages"])
            warn_if_nonzero(label, "language is not null", s["null_language"])
            warn_if_nonzero(label, "bytes > 0", s["non_positive_bytes"])

    # ---- contributors ----
    label = f"{repo}/contributors"
    df = load(label, f"{BRONZE}.raw_{repo}_contributors")
    if df is not None:
        s, ok = run_agg(df, [F.count("*").alias("rows")], [
            F.countDistinct("login").alias("distinct_logins"),
            cnt(F.col("login").isNull()).alias("null_login"),
            cnt(F.col("contributions") < 1).alias("bad_contributions"),
        ], label)
        if s is not None and not_empty(label, s) and ok:
            warn(f"{label}: one row per login", s["rows"] == s["distinct_logins"],
                 f"rows={s['rows']} distinct={s['distinct_logins']}")
            warn_if_nonzero(label, "login is not null", s["null_login"])
            warn_if_nonzero(label, "contributions >= 1", s["bad_contributions"])

    # ---- releases (a repo may have no releases, so empty = WARN only) ----
    label = f"{repo}/releases"
    df = load(label, f"{BRONZE}.raw_{repo}_releases")
    if df is not None:
        s, ok = run_agg(df, [F.count("*").alias("rows")], [
            F.countDistinct("id").alias("distinct_ids"),
            cnt(F.col("id").isNull()).alias("null_id"),
            cnt(F.col("tag_name").isNull()).alias("null_tag"),
            cnt(ts("published_at") > FUTURE_LIMIT).alias("future_published"),
        ], label)
        if s is not None and not_empty(label, s, hard=False) and ok:
            warn(f"{label}: one row per release id", s["rows"] == s["distinct_ids"])
            warn_if_nonzero(label, "id is not null", s["null_id"])
            warn_if_nonzero(label, "tag_name is not null", s["null_tag"])
            warn_if_nonzero(label, "published_at not in the future", s["future_published"])

# COMMAND ----------

# ---------------------------------------------------------------
# Run everything
# ---------------------------------------------------------------
for repo, full in REPOS.items():
    print(f"\n===== {repo} =====")
    check_commits(repo, full)
    check_pull_requests(repo, full)
    check_small_tables(repo, full)

print(f"\nCRITICAL FAILURES: {len(critical_failures)} | WARNINGS: {len(warnings)}")
if warnings:
    print("Warnings (logged only, the job continues):")
    for w in warnings:
        print("  -", w)
if critical_failures:
    raise RuntimeError(
        f"Bronze critical checks failed ({len(critical_failures)}): {critical_failures}")